## 全局变量设置

In [10]:
# 导入必要的模块
import sys
import os

# 将miniconda环境添加到系统路径中
os.environ["PATH"] = f'/root/miniconda3/envs/diffusers/bin:{os.environ["PATH"]}'
# 设置Hugging Face缓存路径
os.environ["HF_HOME"] = ".cache"
# 获取当前工作路径
DB_SCRIPT_WORK_PATH = os.getcwd() 

# 定义训练脚本、转换脚本和反向转换脚本的文件名
TRAINER = "train_dreambooth.py"
CONVERTER = "convert_v3.py"
BACK_CONVERTER = "back_convert.py"

# 定义源模型路径和目标模型路径
SRC_PATH = "./weights-sd"
MODEL_NAME = "./weights-hf"

# 定义模型保存路径
OUTPUT_DIR = "./result"
# 创建输出目录（如果不存在）
!mkdir -p $OUTPUT_DIR

## 转换ckpt文件

将源模型（SafetyNet 模型）转换为 diffusers模型，

In [9]:
# 源模型的检查点路径
SOURCE_CHECKPOINT_PATH = f"./model-sd/model.safetensors"
# 使用转换脚本将 SafetyNet 模型转换为 Hugging Face 模型
# --checkpoint_path 指定源模型的检查点路径
# --dump_path 指定目标 Hugging Face 模型的保存路径
!python $CONVERTER --checkpoint_path $SOURCE_CHECKPOINT_PATH --dump_path $MODEL_NAME

--2023-12-02 09:01:38--  https://raw.githubusercontent.com/CompVis/stable-diffusion/main/configs/stable-diffusion/v1-inference.yaml
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1873 (1.8K) [text/plain]
Saving to: ‘v1-inference.yaml’

v1-inference.yaml   100%[===================>]   1.83K  --.-KB/s    in 0s      

2023-12-02 09:01:38 (35.7 MB/s) - ‘v1-inference.yaml’ saved [1873/1873]

Traceback (most recent call last):
  File "/home/dreambooth/dreambooth/convert_v3.py", line 204, in <module>
    pipe = convert_to_df(load_model(args.checkpoint_path), config_path=args.original_config_file, return_pipe=True, extract_ema=args.extract_ema)
  File "/home/dreambooth/dreambooth/convert_v3.py", line 33, in load_model
    m = safetensors.torch.load_file(path, devic

## 配置dreambooth训练提示词

INSTANCE_PROMPT 中填入 Plaidshirtprogrammer要训练的关键词名。指定任意tag，但是需要找一个“不存在的词”。

CLASS_PROMPT 是让AI自动生成class image用的tag。
复制一份INSTANCE_PROMPT，删掉你学习的tag即可。

同理，下面的预览图tag设置也记得改。

In [12]:
# 实例提示文本
INSTANCE_PROMPT = "Plaidshirtprogrammer"
# 实例数据目录路径
INSTANCE_DIR = "./train-images"

# 类别图片设置
# 类别提示文本
CLASS_PROMPT = "masterpiece, best quality, 1man"
# 负类别提示文本
CLASS_NEGATIVE_PROMPT = "lowres, bad anatomy, bad hands, text, error, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality, normal quality, jpeg artifacts, signature, watermark, username, blurry"
# 类别图片数据目录路径
CLASS_DIR = "./generate-images"

# 预览图标签设置
# 预览图正类别提示文本
SAVE_SAMPLE_PROMPT = "masterpiece, best quality, Plaidshirtprogrammer, looking at viewer"
# 预览图负类别提示文本
SAVE_SAMPLE_NEGATIVE_PROMPT = "lowres, bad anatomy, bad hands, text, error, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality, normal quality, jpeg artifacts, signature, watermark, username, blurry"


## 配置accelerate

该配置文件包含了一些与 Accelerated Inference 相关的默认配置，如计算环境、分布式训练设置、混合精度等。这样的配置文件通常用于定义加速推理的运行环境和选项

In [13]:
!./accelerate.sh

## 设置常用的训练参数

In [14]:
# 最大训练步数
max_train_steps = 1500
# 学习率设置
learning_rate = 5e-6
# 学习率调整策略
# 可选值: ["linear", "cosine", "cosine_with_restarts", "polynomial", "constant", "constant_with_warmup", "cosine_with_restarts_mod", "cosine_mod"]
lr_scheduler = "cosine_with_restarts"
# 学习率预热步数
lr_warmup_steps = 100
# 训练批量大小
train_batch_size = 1
# 自动生成的 class_images 数量
num_class_images = 20
# 是否使用 prior preservation（先验保持）
with_prior_preservation = True
# 是否训练文本编码器
train_text_encoder = False
# 是否使用 ARB（Aspect Ratio Bucket）
use_aspect_ratio_bucket = False
# 从文件名读取 prompt
read_prompt_from_filename = False
# 从 txt 文件读取 prompt
read_prompt_from_txt = False
# 追加提示文本
append_prompt = "instance"
# 保存间隔
save_interval = 500
# 是否使用 deepdanbooru
use_deepdanbooru = False

## 高级参数设置：一般不需要修改
这些参数涉及到训练过程中的一些高级设置，例如图像分辨率、梯度积累步骤、随机数种子、日志记录间隔、梯度剪裁等。这些参数通常用于微调模型的性能或适应具体任务的要求。

In [21]:
# 图像分辨率
resolution = 512
# 梯度积累步骤
gradient_accumulation_steps = 1
# 随机数种子
seed = 1337
# 日志记录间隔
log_interval = 10
# 跳过梯度剪裁
clip_skip = 1
# 生成样本的批量大小
sample_batch_size = 4
# 先验损失的权重
prior_loss_weight = 1.0
# 是否缩放学习率
scale_lr = False
# 是否使用学习率平方根缩放
scale_lr_sqrt = False
# 是否启用梯度检查点
gradient_checkpointing = True
# 是否对 tokens 进行填充
pad_tokens = False
# 调试 ARB（Aspect Ratio Bucket）
debug_arb = False
# 调试提示文本
debug_prompt = False
# 是否使用指数移动平均（EMA）
use_ema = False
# 训练周期数
restart_cycle = 1
# 上一训练周期的结束步数
last_epoch = -1


## 如果是从中途继续训练，需要运行下面这个

如果是继续训练就更改这个路径到想继续训练的模型文件夹然后运行这个

In [24]:
MODEL_NAME = "./result/checkpoint_last"

## 启动训练

设置一些用于训练脚本的命令行参数，传递给训练脚本进行模型训练

In [22]:
# 生成用于训练脚本的命令行参数
ema_arg = "--use_ema" if use_ema else ""
da_arg = "--debug_arb" if debug_arb else ""
db_arg = "--debug_prompt" if debug_prompt else ""
pd_arg = "--pad_tokens" if pad_tokens else ""
gdc_arg = "--gradient_checkpointing" if gradient_checkpointing else ""
dp_arg = "--deepdanbooru" if use_deepdanbooru else "" 
scale_lr_arg = "--scale_lr" if scale_lr else ""
#wandb_arg = "--wandb" if use_wandb else ""
extra_prompt_arg = "--read_prompt_txt" if read_prompt_from_txt else ""
arb_arg = "--use_aspect_ratio_bucket" if use_aspect_ratio_bucket else ""
tte_arg = "--train_text_encoder" if train_text_encoder else ""
ppl_arg = f"--with_prior_preservation --prior_loss_weight={prior_loss_weight}" if with_prior_preservation else ""

# 如果设置了学习率平方根缩放
if scale_lr_sqrt:
  scale_lr_arg = "--scale_lr_sqrt"

# 如果从文件名读取提示文本
if read_prompt_from_filename:
  extra_prompt_arg = "--read_prompt_filename"

使用 Hugging Face Accelerated Inference API 启动指定的 Python 脚本（由变量 $TRAINER 指定）

In [ ]:
!python -m accelerate.commands.launch $TRAINER \
  --mixed_precision="fp16" \
  --pretrained_model_name_or_path=$MODEL_NAME \
  --instance_data_dir=$INSTANCE_DIR \
  --class_data_dir=$CLASS_DIR \
  --output_dir=$OUTPUT_DIR \
  --instance_prompt="{INSTANCE_PROMPT}" \
  --class_prompt="{CLASS_PROMPT}" \
  --class_negative_prompt="{CLASS_NEGATIVE_PROMPT}" \
  --save_sample_prompt="{SAVE_SAMPLE_PROMPT}" \
  --save_sample_negative_prompt="{SAVE_SAMPLE_NEGATIVE_PROMPT}" \
  --seed=$seed \
  --resolution=$resolution \
  --train_batch_size=$train_batch_size \
  --gradient_accumulation_steps=$gradient_accumulation_steps \
  --learning_rate=$learning_rate \
  --lr_scheduler=$lr_scheduler \
  --lr_warmup_steps=$lr_warmup_steps \
  --num_class_images=$num_class_images \
  --sample_batch_size=$sample_batch_size \
  --max_train_steps=$max_train_steps \
  --save_interval=$save_interval \
  --log_interval=$log_interval \
  --clip_skip $clip_skip \
  --num_cycle=$restart_cycle \
  --last_epoch=$last_epoch \
  --append_prompt=$append_prompt \
  --use_8bit_adam --xformers $da_arg $db_arg $ema_arg \
  $ppl_arg $wandb_arg $extra_prompt_arg $gdc_arg $arb_arg $tte_arg $scale_lr_arg $dp_arg $pd_arg

## 转换训练好的模型到ckpt文件

这里需要你修改model_folder_name, 比如
checkpoint_1000
checkpoint_2000
想转换哪个模型写哪个

In [8]:
# 定义模型文件夹名称
model_folder_name = "checkpoint_last"
# 定义转换后的模型路径
convert_model_path = f"output/{model_folder_name}"
# 定义检查点路径
ckpt_path = f'{convert_model_path}/model.ckpt'
# 是否保存为半精度模型（默认为保存单精度模型）
save_half = True
# 根据是否保存半精度模型生成相应的命令行参数
ckpt_convert_half_arg = "--half" if save_half else ""
# 使用 back_convert.py 脚本将模型转换为 TensorFlow 格式
!python back_convert.py --model_path $convert_model_path --checkpoint_path $ckpt_path $ckpt_convert_half_arg
# 打印转换后的模型保存路径
print(f"[*] 转换的模型保存在如下路径 {ckpt_path}")

Traceback (most recent call last):
  File "/home/dreambooth/dreambooth/back_convert.py", line 215, in <module>
    unet_state_dict = torch.load(unet_path, map_location="cpu")
  File "/root/miniconda3/lib/python3.10/site-packages/torch/serialization.py", line 791, in load
    with _open_file_like(f, 'rb') as opened_file:
  File "/root/miniconda3/lib/python3.10/site-packages/torch/serialization.py", line 271, in _open_file_like
    return _open_file(name_or_buffer, mode)
  File "/root/miniconda3/lib/python3.10/site-packages/torch/serialization.py", line 252, in __init__
    super().__init__(open(name, mode))
FileNotFoundError: [Errno 2] No such file or directory: 'output/checkpoint_last/unet/diffusion_pytorch_model.bin'
[*] 转换的模型保存在如下路径 output/checkpoint_last/model.ckpt
